# 02 - Spatial Feature Extraction

## Prerequisites
- `01_face_detection_extraction.ipynb` must be completed
- `preprocessed_faces_integrated/` must contain extracted face images

## Run Order
This notebook can run **in parallel** with `03_frequency` and `04_semantic`.

## Output
- `extracted_features_integrated/spatial/real|fake/.../frame_N.npy` (2048-dim per face)

In [ ]:
!pip install --user timm

In [ ]:
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image

import site, sys
sys.path.append(site.getusersitepackages())
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class Config:
    BASE_DIR = Path(".")
    PREPROCESSED_DIR = BASE_DIR / "preprocessed_faces_integrated"
    FEATURES_DIR = BASE_DIR / "extracted_features_integrated"

config = Config()
(config.FEATURES_DIR / "spatial" / "real").mkdir(parents=True, exist_ok=True)
(config.FEATURES_DIR / "spatial" / "fake").mkdir(parents=True, exist_ok=True)
print("Directories ready")

In [ ]:
class SpatialExtractor(nn.Module):
    """XceptionNet spatial feature extractor (2048-dim)."""

    def __init__(self):
        super().__init__()
        self.model = timm.create_model("xception", pretrained=True, num_classes=0)
        self.model.eval()
        self.transform = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    @torch.no_grad()
    def forward(self, img_path):
        img = Image.open(img_path).convert("RGB")
        img_tensor = self.transform(img).unsqueeze(0).to(device)
        return self.model(img_tensor).squeeze().cpu().numpy()

print("SpatialExtractor (Xception, 2048-dim) defined")

In [ ]:
def extract_single_domain(extractor, extract_fn, domain, preprocessed_dir, output_dir):
    """Extract features for one domain from all preprocessed faces.
    Skips already-extracted features for resumability."""
    image_paths = []
    for label in ["real", "fake"]:
        label_dir = preprocessed_dir / label
        if label_dir.exists():
            for video_dir in label_dir.iterdir():
                if video_dir.is_dir():
                    for img_path in video_dir.glob("*.png"):
                        image_paths.append(img_path)

    print(f"Found {len(image_paths)} face images to process")
    if not image_paths:
        print("ERROR: No images found. Run 01_face_detection_extraction.ipynb first!")
        return 0, 0, 0

    processed = 0
    skipped = 0
    errors = 0

    for img_path in tqdm(image_paths, desc=f"Extracting {domain}"):
        relative_path = img_path.relative_to(preprocessed_dir)
        feature_path = output_dir / domain / relative_path.with_suffix(".npy")

        if feature_path.exists():
            skipped += 1
            continue

        feature_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            features = extract_fn(extractor, img_path)
            np.save(feature_path, features)
            processed += 1
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f"  Error on {img_path.name}: {e}")

    print(f"\n{domain.upper()} complete: Processed={processed}, Skipped={skipped}, Errors={errors}")
    return processed, skipped, errors

In [ ]:
# Initialize and run
extractor = SpatialExtractor().to(device)

processed, skipped, errors = extract_single_domain(
    extractor,
    lambda e, p: e(p),
    "spatial",
    config.PREPROCESSED_DIR,
    config.FEATURES_DIR,
)

del extractor
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Validation

In [ ]:
# ============================================================================
# QUICK VALIDATION
# ============================================================================

print(f"\n{'='*70}")
print("VALIDATION - Spatial Features")
print(f"{'='*70}")

total = 0
for label in ["real", "fake"]:
    feat_dir = config.FEATURES_DIR / "spatial" / label
    if feat_dir.exists():
        count = sum(
            sum(1 for f in os.listdir(d) if f.endswith(".npy"))
            for d in feat_dir.iterdir() if d.is_dir()
        )
        print(f"  {label}: {count} feature files")
        total += count
print(f"  TOTAL: {total} spatial features")

if total > 0:
    # Sanity check: load one and verify shape
    sample = next((config.FEATURES_DIR / "spatial").rglob("*.npy"))
    shape = np.load(sample).shape
    print(f"  Sample shape: {shape} (expected: (2048,))")
    print(f"\nSpatial extraction successful!")
    print("Next: Ensure 03 and 04 are also complete, then run 05.")
else:
    print(f"\nNo spatial features found. Check logs above.")